In [32]:
import itertools
import numpy as np


# Predefine

In [33]:
nx, ny, nz = 5, 5, 5
# v_levels = [0]  # 0 for startpoint, 1 for endpoint, 2 for intermediate point
# --- Sample parameter configuration ---
drone_params = {
    'rho': 1.18,      # kg/m3
    'Cd': 1,
    'Af': 0.25,        # m2
    'm': 4.0  ,          # kg
    'g_acc': np.array([0, 0, -9.81]),
    'A': 0.45,          # m2
    'Pp_hover':65.0  # Watts
}
speed_map = {0: 10.0,1:15.0}
dt = 1
dt_takeoff = 1
dt_landing = 1

In [34]:
Tmax = 35

In [35]:
start_point=[0, 0, 0]
end_point=[4, 4, 4]

In [36]:
def calculate_drone_power(v_i, w_i, a_i, params):
    """
    Calculate total power consumption Pi according to the formula sequence.
    v_i, w_i, a_i are numpy array vectors with shape (3,)
    """
    # Get constants from params
    rho = params['rho']       # Air density
    Cd = params['Cd']         # Drag coefficient
    Af = params['Af']         # Frontal area
    m = params['m']           # Drone weight
    g_acc = params['g_acc']   # Gravitational acceleration vector [0, 0, -9.81]
    A = params['A']           # Rotor disk area
    Pp_hover = params['Pp_hover'] # Profile power when hovering
    mg_scalar = m * np.linalg.norm(g_acc)

    # (2) Velocity relative to wind
    v_a = v_i - w_i
    v_a_norm = np.linalg.norm(v_a)

    # (3) Aerodynamic drag
    Di = -0.5 * rho * Cd * Af * v_a_norm * v_a

    # (4) Thrust force vector
    # print("a_i:", np.array(a_i))
    Ti_vec = m * np.array(a_i) - m * g_acc - Di

    # (5) Magnitude of thrust force
    Ti_mag = np.linalg.norm(Ti_vec)

    # (6) Unit direction vector of thrust
    ti_hat = Ti_vec / Ti_mag if Ti_mag != 0 else np.array([0, 0, 1])

    # (7) Velocity component relative to thrust axis (scalar)
    vc_i = np.dot(v_a, ti_hat)

    # (8) Induced velocity (scalar)
    # Formula: v_ind = -1/2*vc + sqrt((1/2*vc)^2 + Ti/(2*rho*A))
    term1 = -0.5 * vc_i
    term2 = np.sqrt((0.5 * vc_i)**2 + Ti_mag / (2 * rho * A))
    v_ind = term1 + term2

    # (9) Useful power
    Pu_i = np.dot(Ti_vec, v_a)

    # (10) Induced power
    Pind_i = Ti_mag * v_ind

    # (11) Profile power
    Pp_i = Pp_hover * (Ti_mag / mg_scalar)**1.5

    # (12) Total power
    Pi = Pu_i + Pind_i + Pp_i

    return {
        "Pi": Pi,
        "Pu_i": Pu_i,
        "Pind_i": Pind_i,
        "Pp_i": Pp_i,
        "Ti_vec": Ti_vec,
        "Ti_mag": Ti_mag,
    }

In [37]:
def generate_moore_neighbor_dict(nx, ny, nz,speed_map):
    """
    Create dictionary storing Moore Neighbor edges with determined direction.
    Key format: x1_y1_z1_x2_y2_z2
    """
    moore_dict = {}
    
    # 26 directions of Moore neighborhood (all combinations of -1, 0, 1 except (0,0,0))
    directions = list(itertools.product([-1, 0, 1], repeat=3))
    directions.remove((0, 0, 0))
    
    # Iterate through all vertices in x, y, z space
    for x, y, z in itertools.product(range(nx), range(ny), range(nz)):
        for dx, dy, dz in directions:
            nx_val, ny_val, nz_val = x + dx, y + dy, z + dz
            
            # Check boundaries of 3D space
            if 0 <= nx_val < nx and 0 <= ny_val < ny and 0 <= nz_val < nz:
                # Create key in required format: x1_y1_z1_x2_y2_z2
                for _v in speed_map:  # v=0 for startpoint, v=1 for endpoint
                    edge_key = f"{x}_{y}_{z}_{nx_val}_{ny_val}_{nz_val}_{_v}"
                    distance = np.sqrt((nx_val - x)**2 + (ny_val - y)**2 + (nz_val - z)**2)*10
                    speed = speed_map.get(_v, 5.0)

                    edge_vec = np.array([nx_val - x, ny_val - y, nz_val - z], dtype=float)
                    edge_norm = np.linalg.norm(edge_vec)
                    if edge_norm == 0:
                        continue

                    v_i = (speed / edge_norm) * edge_vec
                    power_result = calculate_drone_power(v_i, [1,1,2], [0,0,0], drone_params)

                    moore_dict[edge_key] = {
                        "startpoint": [x, y, z],
                        "endpoint": [nx_val, ny_val, nz_val],
                        "euclidean_distance": distance,
                        "v_level": _v,
                        "v": speed,
                        "v_vector": v_i,
                        "Pi": power_result["Pi"],
                        "Tmaneuver": power_result["Ti_mag"],
                        "Energy":power_result["Pi"]*(distance/speed)
                        
                    }
                    continue
                
    return moore_dict

In [38]:
def calculate_energy_transition(edge_in_key, edge_out_key, wind, params,dt):
    """
    Calculate energy at the intersection point between 2 edges based on Vlevel
    """
    # 1. Decode edge information
    # edge_in_key=space_graph[edge_in_key]
    # edge_out_key=space_graph[edge_out_key]
    p_prev, p_curr, v_in_vec,v_mag_in = edge_in_key["startpoint"], edge_in_key["endpoint"], edge_in_key["v_vector"],edge_in_key["v"]
    _, p_next, v_out_vec,v_mag_out = edge_out_key["startpoint"], edge_out_key["endpoint"], edge_out_key["v_vector"],edge_out_key["v"]

  
    # 3. Calculate v_i and a_i at point p_curr
    # Assume dt is the transition time (average distance / average velocity)
    # dist_avg = (np.linalg.norm(p_curr - p_prev) + np.linalg.norm(p_next - p_curr)) / 2
    # dt = dist_avg / ((v_mag_in + v_mag_out) / 2)

    v_i = (v_in_vec + v_out_vec) / 2
    a_i = (v_out_vec - v_in_vec) / dt

    # --- Physical sequence ---
    rho, Cd, Af, m, A = params['rho'], params['Cd'], params['Af'], params['m'], params['A']
    g_vec = params['g_acc']
    Pp_hover = params['Pp_hover']
    mg_scalar = m * np.linalg.norm(g_vec)

    # (2) Relative velocity
    v_a = v_i - wind
    va_norm = np.linalg.norm(v_a)

    # (3) Drag force Di
    Di = -0.5 * rho * Cd * Af * va_norm * v_a

    # (4) Thrust force Ti (Vector)
    Ti_vec = m * a_i - m * g_vec - Di
    Ti_mag = np.linalg.norm(Ti_vec)

    # (6) Thrust direction
    ti_hat = Ti_vec / Ti_mag if Ti_mag > 1e-6 else np.array([0, 0, 1])

    # (7) Velocity along thrust axis
    vc_i = np.dot(v_a, ti_hat)

    # (8) Induced velocity
    v_ind = -0.5 * vc_i + np.sqrt(max(0, (0.5 * vc_i)**2 + Ti_mag / (2 * rho * A)))

    # (9-12) Power components
    Pu_i = np.dot(Ti_vec, v_a)
    Pind_i = Ti_mag * v_ind
    Pp_i = Pp_hover * (Ti_mag / mg_scalar)**1.5
    Pi = Pu_i + Pind_i + Pp_i

    return {
        # "edge_in": edge_in_key,
        # "edge_out": edge_out_key,
        "total_power_Pi": Pi,
        "Tmaneuver": Ti_mag,
        # "acceleration": a_i
    }

# calculate the coeffiction

In [39]:
w_i = np.array([1, 1, 2])
space_graph = generate_moore_neighbor_dict(nx, ny, nz,speed_map)

print(f"\nDirected Edge Dictionary: {len(space_graph)}")


Directed Edge Dictionary: 4144


In [61]:
def tinh_lambda(x, y, z):
    term1 = 28 + 22 * (x + y + z - 6)
    
    term2 = 17 * (
        (x - 2) * (y - 2) + 
        (x - 2) * (z - 2) + 
        (y - 2) * (z - 2)
    )
    
    term3 = 13 * (x - 2) * (y - 2) * (z - 2)
    
    lamda_val = term1 + term2 + term3
    return lamda_val

# Ví dụ sử dụng:ny
ket_qua = tinh_lambda(300, 300, 140)
print(f"Recheck number of Directed Edge: {ket_qua*2*len(speed_map)}")


Recheck number of Directed Edge: 648953744


In [41]:

# Drone flies from (1,1,1) -> (2,2,2) with Level 1, then continues to (3,2,2) with Level 0
edge1 = "1_1_1_2_2_2_0"
edge2 = "2_2_2_3_2_2_0"
wind_vec = np.array([1.0, 0, 0]) # Wind blowing along X axis

edge1_data = space_graph[edge1]
edge2_data = space_graph[edge2]
result = calculate_energy_transition(edge1_data, edge2_data, wind_vec, drone_params, dt) # Transition time
print(f"\nEnergy transition from {edge1} to {edge2}: ")
print(f"{result['total_power_Pi']:.2f} J, Thrust force={result['Tmaneuver']:.3f} N")


Energy transition from 1_1_1_2_2_2_0 to 2_2_2_3_2_2_0: 
384.27 J, Thrust force=37.374 N


In [42]:
space_graph

{'0_0_0_0_0_1_0': {'startpoint': [0, 0, 0],
  'endpoint': [0, 0, 1],
  'euclidean_distance': 10.0,
  'v_level': 0,
  'v': 10.0,
  'v_vector': array([ 0.,  0., 10.]),
  'Pi': 672.0567504861731,
  'Tmaneuver': 48.855765014121715,
  'Energy': 672.0567504861731},
 '0_0_0_0_0_1_1': {'startpoint': [0, 0, 0],
  'endpoint': [0, 0, 1],
  'euclidean_distance': 10.0,
  'v_level': 1,
  'v': 15.0,
  'v_vector': array([ 0.,  0., 15.]),
  'Pi': 1210.4221164484522,
  'Tmaneuver': 64.37238584583366,
  'Energy': 806.9480776323014},
 '0_0_0_0_1_0_0': {'startpoint': [0, 0, 0],
  'endpoint': [0, 1, 0],
  'euclidean_distance': 10.0,
  'v_level': 0,
  'v': 10.0,
  'v_vector': array([ 0., 10.,  0.]),
  'Pi': 315.93758672197987,
  'Tmaneuver': 38.54851124284415,
  'Energy': 315.93758672197987},
 '0_0_0_0_1_0_1': {'startpoint': [0, 0, 0],
  'endpoint': [0, 1, 0],
  'euclidean_distance': 10.0,
  'v_level': 1,
  'v': 15.0,
  'v_vector': array([ 0., 15.,  0.]),
  'Pi': 597.9564674711731,
  'Tmaneuver': 45.72221863

In [43]:
# Filter all edges starting from point (0, 0, 0)
edges_from_origin = {key: value for key, value in space_graph.items() if value['startpoint'] == start_point}

print(f"Number of edges starting from (0,0,0): {len(edges_from_origin)}")
print("\nEdges from origin:")
for key, data in edges_from_origin.items():
    astart = data['v_vector']/dt_takeoff
    Etakeoff = calculate_drone_power(data['v_vector']/2, [1,1,2], astart, drone_params)["Pi"]*dt_takeoff
    space_graph[key]["Etakeoff"] = Etakeoff  # Add takeoff energy to the edge
    print(f"{key}:   Energy={data['Energy']:.2f}, Thrust force={data['Tmaneuver']:.3f} N, Etakeoff={Etakeoff:.2f} J")

Number of edges starting from (0,0,0): 14

Edges from origin:
0_0_0_0_0_1_0:   Energy=672.06, Thrust force=48.856 N, Etakeoff=1027.33 J
0_0_0_0_0_1_1:   Energy=806.95, Thrust force=64.372 N, Etakeoff=1632.48 J
0_0_0_0_1_0_0:   Energy=315.94, Thrust force=38.549 N, Etakeoff=583.27 J
0_0_0_0_1_0_1:   Energy=398.64, Thrust force=45.722 N, Etakeoff=1027.02 J
0_0_0_0_1_1_0:   Energy=776.48, Thrust force=45.779 N, Etakeoff=887.58 J
0_0_0_0_1_1_1:   Energy=936.15, Thrust force=58.635 N, Etakeoff=1431.56 J
0_0_0_1_0_0_0:   Energy=315.94, Thrust force=38.549 N, Etakeoff=583.27 J
0_0_0_1_0_0_1:   Energy=398.64, Thrust force=45.722 N, Etakeoff=1027.02 J
0_0_0_1_0_1_0:   Energy=776.48, Thrust force=45.779 N, Etakeoff=887.58 J
0_0_0_1_0_1_1:   Energy=936.15, Thrust force=58.635 N, Etakeoff=1431.56 J
0_0_0_1_1_0_0:   Energy=430.31, Thrust force=38.303 N, Etakeoff=567.67 J
0_0_0_1_1_0_1:   Energy=525.84, Thrust force=44.671 N, Etakeoff=995.11 J
0_0_0_1_1_1_0:   Energy=850.55, Thrust force=44.220 N, E

In [44]:
# Filter all edges ending at point (4, 4, 5)
edges_to_destination = {key: value for key, value in space_graph.items() if value['endpoint'] == end_point}

print(f"Number of edges ending at (4,4,5): {len(edges_to_destination)}")
print("\nEdges to destination:")
for key, data in edges_to_destination.items():
    astart = data['v_vector']/dt_landing
    Elanding = calculate_drone_power(data['v_vector']/2, [1,1,2], astart, drone_params)["Pi"]*dt_landing
    space_graph[key]["Elanding"] = Elanding  # Add landing energy to the edge
    print(f"{key}: {data['endpoint']}  Energy={data['Energy']:.2f}, Thrust force={data['Tmaneuver']:.3f} N, Elanding={Elanding:.2f} J")

Number of edges ending at (4,4,5): 14

Edges to destination:
3_3_3_4_4_4_0: [4, 4, 4]  Energy=850.55, Thrust force=44.220 N, Elanding=819.48 J
3_3_3_4_4_4_1: [4, 4, 4]  Energy=1019.87, Thrust force=55.581 N, Elanding=1331.00 J
3_3_4_4_4_4_0: [4, 4, 4]  Energy=430.31, Thrust force=38.303 N, Elanding=567.67 J
3_3_4_4_4_4_1: [4, 4, 4]  Energy=525.84, Thrust force=44.671 N, Elanding=995.11 J
3_4_3_4_4_4_0: [4, 4, 4]  Energy=776.48, Thrust force=45.779 N, Elanding=887.58 J
3_4_3_4_4_4_1: [4, 4, 4]  Energy=936.15, Thrust force=58.635 N, Elanding=1431.56 J
3_4_4_4_4_4_0: [4, 4, 4]  Energy=315.94, Thrust force=38.549 N, Elanding=583.27 J
3_4_4_4_4_4_1: [4, 4, 4]  Energy=398.64, Thrust force=45.722 N, Elanding=1027.02 J
4_3_3_4_4_4_0: [4, 4, 4]  Energy=776.48, Thrust force=45.779 N, Elanding=887.58 J
4_3_3_4_4_4_1: [4, 4, 4]  Energy=936.15, Thrust force=58.635 N, Elanding=1431.56 J
4_3_4_4_4_4_0: [4, 4, 4]  Energy=315.94, Thrust force=38.549 N, Elanding=583.27 J
4_3_4_4_4_4_1: [4, 4, 4]  Energy

#  Eliminate variable

In [45]:
# Eliminate edges that end at start_point or start at end_point
edges_to_start = {
    key: data
    for key, data in space_graph.items()
    if data["endpoint"] == start_point
}

edges_from_end = {
    key: data
    for key, data in space_graph.items()
    if data["startpoint"] == end_point
}

print(edges_to_start)
print(edges_from_end)

{'0_0_1_0_0_0_0': {'startpoint': [0, 0, 1], 'endpoint': [0, 0, 0], 'euclidean_distance': 10.0, 'v_level': 0, 'v': 10.0, 'v_vector': array([  0.,   0., -10.]), 'Pi': 43.80713058250499, 'Tmaneuver': 18.030051082189633, 'Energy': 43.80713058250499}, '0_0_1_0_0_0_1': {'startpoint': [0, 0, 1], 'endpoint': [0, 0, 0], 'euclidean_distance': 10.0, 'v_level': 1, 'v': 15.0, 'v_vector': array([  0.,   0., -15.]), 'Pi': 69.8693144973135, 'Tmaneuver': 5.015632022506374, 'Energy': 46.57954299820899}, '0_1_0_0_0_0_0': {'startpoint': [0, 1, 0], 'endpoint': [0, 0, 0], 'euclidean_distance': 10.0, 'v_level': 0, 'v': 10.0, 'v_vector': array([  0., -10.,   0.]), 'Pi': 389.53117003560425, 'Tmaneuver': 40.31505623874403, 'Energy': 389.53117003560425}, '0_1_0_0_0_0_1': {'startpoint': [0, 1, 0], 'endpoint': [0, 0, 0], 'euclidean_distance': 10.0, 'v_level': 1, 'v': 15.0, 'v_vector': array([  0., -15.,   0.]), 'Pi': 818.9326025625257, 'Tmaneuver': 51.456870888185904, 'Energy': 545.9550683750172}, '0_1_1_0_0_0_0':

In [46]:
# Eliminate edge in obstacle

In [47]:
# Eliminate edge with thrust force above a certain threshold (e.g., 15 N)

In [48]:
# Xoá các cạnh trong space_graph có key nằm trong edges_to_start hoặc edges_from_end
keys_to_remove = set(edges_to_start.keys()) | set(edges_from_end.keys())

removed = 0
for k in keys_to_remove:
    if k in space_graph:
        del space_graph[k]
        removed += 1

print(f"Đã xoá {removed} key khỏi space_graph")
print(f"Số cạnh còn lại: {len(space_graph)}")

Đã xoá 28 key khỏi space_graph
Số cạnh còn lại: 4116


# QUBO matrix creation

In [49]:
# Create QUBO matrix from space_graph
edge_keys = list(space_graph.keys())
N = len(edge_keys)
edge_index = {key: idx for idx, key in enumerate(edge_keys)}

print(f"Number of edges (N): {N}")
print(f"QUBO matrix size: {N} x {N}")

# Initialize QUBO matrix
Q = np.zeros((N, N), dtype=float)


Number of edges (N): 4116
QUBO matrix size: 4116 x 4116


## Objective function

In [50]:
# Gán giá trị đường chéo Q[i,i] = 3 cho các cạnh có startpoint
list_edges_start = {
    key: space_graph[key]
    for key in edge_index
    if space_graph[key]["startpoint"] == start_point
}
max_energy_start = 0
for key in list_edges_start:
    idx = edge_index[key]  # lấy index của key trong graph
    bufVal= 0.5*space_graph[key]["Energy"] + space_graph[key].get("Etakeoff", 0.0)
    Q[idx, idx] += bufVal
    if bufVal > max_energy_start:
        max_energy_start = bufVal

print(f"Max energy for edges from start point: {max_energy_start:.2f} J")


Max energy for edges from start point: 2035.95 J


In [51]:
# Gán giá trị đường chéo Q[i,i] = 3 cho các cạnh có endpoint
list_edges_end = {
    key: space_graph[key]
    for key in edge_index
    if space_graph[key]["endpoint"] == end_point
}
max_energy_end = 0

for key in list_edges_end:
    idx = edge_index[key]  # lấy index của key trong graph
    bufVal = 0.5*space_graph[key]["Energy"] + space_graph[key].get("Elanding", 0.0)
    Q[idx, idx] += bufVal
    if bufVal > max_energy_end:
        max_energy_end = bufVal
print(f"Max energy for edges to end point: {max_energy_end:.2f} J")

Max energy for edges to end point: 2035.95 J


In [52]:
# Lặp qua toàn bộ point trong không gian, trừ start và end
start_t = tuple(start_point)
end_t = tuple(end_point)

all_mid_points = []
for x, y, z in itertools.product(range(nx), range(ny), range(nz)):
    p = (x, y, z)
    if p == start_t or p == end_t:
        continue
    all_mid_points.append(p)



In [53]:
max_pair_energy = 0
for  target_xyz in all_mid_points:
    prefix = f"{target_xyz[0]}_{target_xyz[1]}_{target_xyz[2]}_"
    edges_out_list= {
        k: v
        for k, v in space_graph.items()
        if tuple(v["startpoint"]) == target_xyz
    }
    edges_in_list= {
        k: v
        for k, v in space_graph.items()
        if tuple(v["endpoint"]) == target_xyz
    }
    # print(f"So key bat dau bang {target_xyz}: {len(edges_prefix)}")
    for edges_in in edges_in_list:
        idx_in = edge_index[edges_in]
        for edges_out in edges_out_list:
            idx_out = edge_index[edges_out]
            change_dir = calculate_energy_transition(space_graph[edges_in], space_graph[edges_out], wind_vec, drone_params, dt)
            bufVal = 0.5*space_graph[edges_in]["Energy"] + 0.5*space_graph[edges_out]["Energy"] + change_dir["total_power_Pi"]
            Q[idx_in, idx_out] += bufVal
            if bufVal > max_pair_energy:
                max_pair_energy = bufVal

max_pair_energy

3018.00057797481

## Constraint

In [54]:
for key in list_edges_start:
    idx = edge_index[key]  # lấy index của key trong graph
    Q[idx, idx] -= max_energy_start

for i in range(len(list_edges_start)):
    for j in range(i+1, len(list_edges_start)):
        idx_i = edge_index[list(list_edges_start.keys())[i]]
        idx_j = edge_index[list(list_edges_start.keys())[j]]

        Q[idx_i][idx_j] += 2*max_energy_start

In [55]:
for key in list_edges_end:
    idx = edge_index[key]  # lấy index của key trong graph
    Q[idx, idx] -= max_energy_end

for i in range(len(list_edges_end)):
    for j in range(i+1, len(list_edges_end)):
        idx_i = edge_index[list(list_edges_end.keys())[i]]
        idx_j = edge_index[list(list_edges_end.keys())[j]]

        Q[idx_i][idx_j] += 2*max_energy_end

In [56]:
max_pair_energy = 0
for  target_xyz in all_mid_points:
    prefix = f"{target_xyz[0]}_{target_xyz[1]}_{target_xyz[2]}_"
    edges_out_list= {
        k: v
        for k, v in space_graph.items()
        if tuple(v["startpoint"]) == target_xyz
    }
    edges_in_list= {
        k: v
        for k, v in space_graph.items()
        if tuple(v["endpoint"]) == target_xyz
    }
    for edges_in in edges_in_list:
        idx_in = edge_index[edges_in]
        Q[idx_in, idx_in] += max_pair_energy

    for edges_out in edges_out_list:
        idx_out = edge_index[edges_out]
        Q[idx_out, idx_out] += max_pair_energy

    for i in range(len(edges_in_list)):
        for j in range(i+1, len(edges_in_list)):
            idx_i = edge_index[list(edges_in_list.keys())[i]]
            idx_j = edge_index[list(edges_in_list.keys())[j]]

            Q[idx_i][idx_j] += 2*max_pair_energy

    for i in range(len(edges_out_list)):
        for j in range(i+1, len(edges_out_list)):
            idx_i = edge_index[list(edges_out_list.keys())[i]]
            idx_j = edge_index[list(edges_out_list.keys())[j]]

            Q[idx_i][idx_j] += 2*max_pair_energy

    for edges_in in edges_in_list:
        idx_in = edge_index[edges_in]
        for edges_out in edges_out_list:
            idx_out = edge_index[edges_out]
            Q[idx_in, idx_out] +=-2*max_pair_energy

In [57]:
for key in edge_index:
    if space_graph[key]["Tmaneuver"] > Tmax:  # Threshold for high thrust
        idx = edge_index[key]
        Q[idx, idx] += max_pair_energy

In [58]:
for  target_xyz in all_mid_points:
    prefix = f"{target_xyz[0]}_{target_xyz[1]}_{target_xyz[2]}_"
    edges_out_list= {
        k: v
        for k, v in space_graph.items()
        if tuple(v["startpoint"]) == target_xyz
    }
    edges_in_list= {
        k: v
        for k, v in space_graph.items()
        if tuple(v["endpoint"]) == target_xyz
    }
    # print(f"So key bat dau bang {target_xyz}: {len(edges_prefix)}")
    for edges_in in edges_in_list:
        idx_in = edge_index[edges_in]
        for edges_out in edges_out_list:
            idx_out = edge_index[edges_out]
            change_dir = calculate_energy_transition(space_graph[edges_in], space_graph[edges_out], wind_vec, drone_params, dt)
            if change_dir["Tmaneuver"] > Tmax:
                Q[idx_in, idx_out] += max_pair_energy

In [59]:
Q

array([[ -672.59744566,  4071.90353852,  4071.90353852, ...,
            0.        ,     0.        ,     0.        ],
       [    0.        ,     0.        ,  4071.90353852, ...,
            0.        ,     0.        ,     0.        ],
       [    0.        ,     0.        , -1294.7149282 , ...,
            0.        ,     0.        ,     0.        ],
       ...,
       [    0.        ,     0.        ,     0.        , ...,
            0.        ,     0.        ,     0.        ],
       [    0.        ,     0.        ,     0.        , ...,
            0.        ,  -672.59744566,  4071.90353852],
       [    0.        ,     0.        ,     0.        , ...,
            0.        ,     0.        ,     0.        ]])

In [60]:

print(f"\nQUBO Matrix Q shape: {Q.shape}")
print(f"Non-zero elements: {np.count_nonzero(Q)}")
print(f"Diagonal sample (first 5): {np.diag(Q)[:5]}")


QUBO Matrix Q shape: (4116, 4116)
Non-zero elements: 153360
Diagonal sample (first 5): [ -672.59744566     0.         -1294.7149282   -809.61070775
  -760.12963139]
